[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/somanshusingla/llm-from-scratch/blob/main/notebooks/06_ch7_instruction_finetuning.ipynb)

# Chapter 7 — Fine-Tuning to Follow Instructions

**Build a Large Language Model (From Scratch)** · notebook 6 of 7

This is how a raw text-completion model becomes a helpful **assistant**. We take
pretrained GPT-2 and fine-tune it on `(instruction, response)` pairs so it learns
to *follow instructions* — the core idea behind ChatGPT-style models
(instruction / supervised fine-tuning, SFT).

The plan:

1. Load an **instruction dataset** of `{instruction, input, output}` examples.
2. Format each example with a consistent **prompt template** (Alpaca style).
3. Batch with a **custom collate function** that pads and masks the loss on
   padding tokens.
4. **Fully fine-tune** pretrained GPT-2 on these pairs.
5. Generate responses to unseen instructions and inspect the results.

> **CPU note:** real instruction tuning uses a bigger model (the book uses GPT-2
> *medium*, 355M) and all the data for several epochs. To keep this runnable on a
> laptop CPU we use GPT-2 **124M**, a **subset** of the data, and **2 epochs**.
> The results will be rough but clearly show the model learning the format and
> starting to follow instructions. Scale up on a Colab GPU for real quality.

## 0. Setup

In [ ]:
import importlib, subprocess, sys
def ensure(pkg, name=None):
    try: importlib.import_module(name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
ensure("tiktoken")

import torch, torch.nn as nn, tiktoken, os, json, urllib.request, functools, time
from torch.utils.data import Dataset, DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tiktoken.get_encoding("gpt2")
print("torch:", torch.__version__, "| device:", device)

## Recap: the GPT model + pretrained-weight loader (from Chapters 4-5)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out, self.num_heads, self.head_dim = d_out, num_heads, d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
    def forward(self, x):
        b, n, _ = x.shape
        k = self.W_key(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        q = self.W_query(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_value(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        scores = q @ k.transpose(2, 3)
        scores.masked_fill_(self.mask.bool()[:n, :n], -torch.inf)
        w = self.dropout(torch.softmax(scores / k.shape[-1]**0.5, dim=-1))
        return self.out_proj((w @ v).transpose(1, 2).contiguous().view(b, n, self.d_out))

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim)); self.shift = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True); var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift

class GELU(nn.Module):
    def forward(self, x):
        return 0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi))*(x+0.044715*torch.pow(x,3))))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]), GELU(),
                                    nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"]))
    def forward(self, x): return self.layers(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"], cfg["context_length"],
                                      cfg["drop_rate"], cfg["n_heads"], cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"]); self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
    def forward(self, x):
        x = x + self.drop_shortcut(self.att(self.norm1(x)))
        x = x + self.drop_shortcut(self.ff(self.norm2(x)))
        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
    def forward(self, in_idx):
        b, seq_len = in_idx.shape
        x = self.tok_emb(in_idx) + self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        return self.out_head(self.final_norm(self.trf_blocks(self.drop_emb(x))))

def _copy(dst, src):
    assert dst.shape == src.shape, f"{tuple(dst.shape)} vs {tuple(src.shape)}"
    with torch.no_grad(): dst.copy_(src)

def load_weights_from_hf(gpt, hf_model):
    sd = hf_model.state_dict()
    _copy(gpt.tok_emb.weight, sd["transformer.wte.weight"]); _copy(gpt.pos_emb.weight, sd["transformer.wpe.weight"])
    for b in range(len(gpt.trf_blocks)):
        p, blk = f"transformer.h.{b}.", gpt.trf_blocks[b]
        w = sd[p+"attn.c_attn.weight"]; q,k,v = w.split(w.shape[1]//3, dim=1)
        _copy(blk.att.W_query.weight, q.T); _copy(blk.att.W_key.weight, k.T); _copy(blk.att.W_value.weight, v.T)
        bq,bk,bv = sd[p+"attn.c_attn.bias"].split(sd[p+"attn.c_attn.bias"].shape[0]//3, dim=0)
        _copy(blk.att.W_query.bias, bq); _copy(blk.att.W_key.bias, bk); _copy(blk.att.W_value.bias, bv)
        _copy(blk.att.out_proj.weight, sd[p+"attn.c_proj.weight"].T); _copy(blk.att.out_proj.bias, sd[p+"attn.c_proj.bias"])
        _copy(blk.norm1.scale, sd[p+"ln_1.weight"]); _copy(blk.norm1.shift, sd[p+"ln_1.bias"])
        _copy(blk.norm2.scale, sd[p+"ln_2.weight"]); _copy(blk.norm2.shift, sd[p+"ln_2.bias"])
        _copy(blk.ff.layers[0].weight, sd[p+"mlp.c_fc.weight"].T); _copy(blk.ff.layers[0].bias, sd[p+"mlp.c_fc.bias"])
        _copy(blk.ff.layers[2].weight, sd[p+"mlp.c_proj.weight"].T); _copy(blk.ff.layers[2].bias, sd[p+"mlp.c_proj.bias"])
    _copy(gpt.final_norm.scale, sd["transformer.ln_f.weight"]); _copy(gpt.final_norm.shift, sd["transformer.ln_f.bias"])
    _copy(gpt.out_head.weight, sd["transformer.wte.weight"])

def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(idx[:, -context_size:])[:, -1, :]
        if top_k is not None:
            tv, _ = torch.topk(logits, top_k)
            logits = torch.where(logits < tv[:, -1], torch.tensor(float("-inf")).to(logits.device), logits)
        if temperature > 0.0:
            idx_next = torch.multinomial(torch.softmax(logits/temperature, dim=-1), num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if eos_id is not None and (idx_next == eos_id).all(): break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

def text_to_token_ids(text, tk): return torch.tensor(tk.encode(text, allowed_special={"<|endoftext|>"})).unsqueeze(0)
def token_ids_to_text(ids, tk): return tk.decode(ids.squeeze(0).tolist())
print("Model code ready.")

## 7.1 & 7.2 The instruction dataset

Each example has an `instruction`, an optional `input`, and the desired `output`.
We download ~1,100 such pairs. (Falls back to a tiny built-in set if offline.)

In [ ]:
def load_instruction_data():
    url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/"
           "ch07/01_main-chapter-code/instruction-data.json")
    path = "instruction-data.json"
    try:
        if not os.path.exists(path):
            urllib.request.urlretrieve(url, path)
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print("Loaded instruction dataset:", len(data), "examples")
        return data
    except Exception as e:
        print("Download failed:", repr(e), "-- using a small built-in set.")
        base = [
            {"instruction": "Convert the active sentence to passive.", "input": "The chef cooks the meal.",
             "output": "The meal is cooked by the chef."},
            {"instruction": "What is the capital of France?", "input": "", "output": "The capital of France is Paris."},
            {"instruction": "Give an antonym of 'happy'.", "input": "", "output": "An antonym of 'happy' is 'sad'."},
            {"instruction": "Rewrite the sentence to be more formal.", "input": "Gonna be late.",
             "output": "I will be arriving late."},
            {"instruction": "Name the largest planet in our solar system.", "input": "",
             "output": "The largest planet in our solar system is Jupiter."},
            {"instruction": "Translate 'good morning' to Spanish.", "input": "",
             "output": "'Good morning' in Spanish is 'buenos dias'."},
            {"instruction": "What is 12 multiplied by 8?", "input": "", "output": "12 multiplied by 8 is 96."},
            {"instruction": "Correct the spelling.", "input": "recieve", "output": "The correct spelling is 'receive'."},
        ]
        return base * 40  # ~320 examples so training has something to chew on

data = load_instruction_data()

In [ ]:
def format_input(entry):
    instruction_text = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}")
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

# Show one fully formatted training example
example = data[0]
print(format_input(example) + f"\n\n### Response:\n{example['output']}")

### Subset + split (for a fast CPU run)

We take a subset and split into train / validation / test.

In [ ]:
SUBSET = len(data) if device.type == "cuda" else min(len(data), 400)  # all data on GPU
data = data[:SUBSET]

train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.10)
train_data = data[:train_portion]
test_data  = data[train_portion:train_portion + test_portion]
val_data   = data[train_portion + test_portion:]
print(f"train {len(train_data)} | val {len(val_data)} | test {len(test_data)}")

## 7.3 & 7.4 Batching: dataset + custom collate

The dataset stores each example as a single token sequence
(`prompt + response`). The **collate function** pads a batch to equal length,
builds targets (inputs shifted by one), and replaces padding in the targets with
`-100` so it's **ignored by the loss** (cross-entropy skips `-100`).

In [ ]:
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            full = format_input(entry) + f"\n\n### Response:\n{entry['output']}"
            self.encoded_texts.append(tokenizer.encode(full))
    def __getitem__(self, i): return self.encoded_texts[i]
    def __len__(self): return len(self.data)

def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None):
    batch_max = max(len(item) + 1 for item in batch)   # +1 for the appended eot
    inputs_lst, targets_lst = [], []
    for item in batch:
        new = item + [pad_token_id]
        padded = new + [pad_token_id] * (batch_max - len(new))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        mask = targets == pad_token_id
        idx = torch.nonzero(mask).squeeze()
        if idx.numel() > 1:                            # keep first eot, ignore the rest
            targets[idx[1:]] = ignore_index
        if allowed_max_length is not None:
            inputs, targets = inputs[:allowed_max_length], targets[:allowed_max_length]
        inputs_lst.append(inputs); targets_lst.append(targets)
    return torch.stack(inputs_lst), torch.stack(targets_lst)

# quick demo of the collate on 2 tiny sequences
demo = [[1, 2, 3], [4, 5]]
di, dt = custom_collate_fn(demo)
print("inputs:\n", di)
print("targets (-100 = ignored):\n", dt)

In [ ]:
ALLOWED_MAX_LENGTH = 512 if device.type == "cuda" else 256   # longer sequences on GPU
collate = functools.partial(custom_collate_fn, allowed_max_length=ALLOWED_MAX_LENGTH)

BATCH = 8
train_loader = DataLoader(InstructionDataset(train_data, tokenizer), batch_size=BATCH,
                          collate_fn=collate, shuffle=True, drop_last=True, num_workers=0)
val_loader   = DataLoader(InstructionDataset(val_data, tokenizer), batch_size=BATCH,
                          collate_fn=collate, shuffle=False, drop_last=False, num_workers=0)
print("train batches:", len(train_loader), "| val batches:", len(val_loader))

## 7.5 Load the pretrained LLM

We start from pretrained GPT-2 124M (the book uses the 355M "medium" model for
better quality — try it on a GPU). Falls back to a small model if offline.

In [ ]:
GPT2_124M = {"vocab_size":50257, "context_length":1024, "emb_dim":768,
             "n_heads":12, "n_layers":12, "drop_rate":0.0, "qkv_bias":True}
import os, warnings
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
warnings.filterwarnings("ignore")

# On GPU we use the larger GPT-2 medium (355M) -- the same size the book uses -- for
# noticeably better instruction-following. On CPU we use GPT-2 124M to stay fast.
if device.type == "cuda":
    HF_NAME = "gpt2-medium"
    GPT2_124M = {"vocab_size":50257, "context_length":1024, "emb_dim":1024,
                 "n_heads":16, "n_layers":24, "drop_rate":0.0, "qkv_bias":True}
else:
    HF_NAME = "gpt2"
    GPT2_124M = {"vocab_size":50257, "context_length":1024, "emb_dim":768,
                 "n_heads":12, "n_layers":12, "drop_rate":0.0, "qkv_bias":True}

model = None
try:
    from transformers import GPT2LMHeadModel
    try:
        from transformers.utils import logging as hf_logging
        hf_logging.disable_progress_bar(); hf_logging.set_verbosity_error()
    except Exception:
        pass
    print(f"Loading pretrained {HF_NAME} ...")
    hf_model = GPT2LMHeadModel.from_pretrained(HF_NAME); hf_model.eval()
    model = GPTModel(GPT2_124M); load_weights_from_hf(model, hf_model)
    print(f"Pretrained {HF_NAME} loaded ({sum(p.numel() for p in model.parameters()):,} params).")
except Exception as e:
    print("Pretrained load failed:", repr(e), "-- using a small model.")
    GPT2_124M = {**GPT2_124M, "emb_dim":256, "n_heads":8, "n_layers":6}
    torch.manual_seed(123); model = GPTModel(GPT2_124M)
model.to(device)

# See what the model says BEFORE instruction tuning
sample = val_data[0] if len(val_data) else test_data[0]
prompt = format_input(sample)
out = generate(model, text_to_token_ids(prompt, tokenizer).to(device),
               max_new_tokens=35, context_size=GPT2_124M["context_length"], eos_id=50256)
print("BEFORE fine-tuning:\n", token_ids_to_text(out, tokenizer)[len(prompt):].strip()[:200])

## 7.6 Fine-tune on instruction data

Unlike classification, here we do **full** fine-tuning (all weights train) using
the same next-token loss — but only on these instruction/response pairs. The
`-100` masking means the model is scored on producing the right *response*.

In [ ]:
def calc_loss_batch(inp, tgt, model, device):
    inp, tgt = inp.to(device), tgt.to(device)
    logits = model(inp)
    return torch.nn.functional.cross_entropy(logits.flatten(0, 1), tgt.flatten())  # ignores -100

def calc_loss_loader(loader, model, device, num_batches=None):
    n = len(loader) if num_batches is None else min(num_batches, len(loader))
    if n == 0: return float("nan")
    total = 0.0
    for i, (inp, tgt) in enumerate(loader):
        if i >= n: break
        total += calc_loss_batch(inp, tgt, model, device).item()
    return total / n

def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter):
    train_losses, val_losses = [], []
    global_step = -1
    for epoch in range(num_epochs):
        model.train()
        for inp, tgt in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(inp, tgt, model, device)
            loss.backward()
            optimizer.step()
            global_step += 1
            if global_step % eval_freq == 0:
                model.eval()
                tr = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
                vl = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
                model.train()
                train_losses.append(tr); val_losses.append(vl)
                print(f"Ep {epoch+1} (step {global_step:04d}): train loss {tr:.3f}, val loss {vl:.3f}")
    return train_losses, val_losses

torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
t0 = time.time()
NUM_EPOCHS = 3 if device.type == "cuda" else 2   # more epochs on GPU
train_model_simple(model, train_loader, val_loader, optimizer, device,
                   num_epochs=NUM_EPOCHS, eval_freq=10, eval_iter=5)
print(f"Instruction fine-tuning took {time.time()-t0:.1f}s on {device}")

## 7.7 & 7.8 Generate and inspect responses

Let's ask the fine-tuned model to complete a few held-out instructions and
compare to the reference answers. With 124M + a small subset + 1 epoch the output
is imperfect, but you'll see it adopt the response format and often get the gist.

In [ ]:
def extract_response(full_text, prompt):
    return full_text[len(prompt):].replace("### Response:", "").strip()

model.eval()
for entry in test_data[:3]:
    prompt = format_input(entry)
    out = generate(model, text_to_token_ids(prompt, tokenizer).to(device),
                   max_new_tokens=40, context_size=GPT2_124M["context_length"], eos_id=50256)
    full = token_ids_to_text(out, tokenizer)
    print("INSTRUCTION:", entry["instruction"])
    if entry["input"]: print("INPUT:      ", entry["input"])
    print("EXPECTED:   ", entry["output"])
    print("MODEL:      ", extract_response(full, prompt)[:200])
    print("-" * 70)

### A note on automated evaluation

The book scores responses automatically using another LLM (e.g. Llama 3 via
Ollama) as a judge. That needs a separate large model, so we skip it here and
evaluate qualitatively by reading the outputs above. The takeaway: after SFT the
model responds in the instruction/response *format* and follows simple
instructions — exactly the behavior we trained for.

## 7.9 Conclusions — you built an LLM end to end! 🎉

Across these notebooks you went from raw text to a working, instruction-following
language model:

1. **Text data** → tokens, BPE, embeddings, positions.
2. **Attention** → scaled dot-product, causal, multi-head.
3. **GPT model** → transformer blocks, layer norm, GELU, generation.
4. **Pretraining** → loss, training loop, sampling, loading OpenAI weights.
5. **Classification fine-tuning** → transfer learning into a spam detector.
6. **Instruction fine-tuning** → an assistant that follows instructions.

### Try it yourself
- On a Colab GPU: use GPT-2 medium (355M), remove `SUBSET`, and train 2-3 epochs.
- Save the fine-tuned model with `torch.save(model.state_dict(), "sft.pth")`.
- Write your own instructions and see how it responds.

**Next:** `07_appendix_extras.ipynb` — optional upgrades (LoRA, learning-rate
warmup, gradient clipping).